# 재개 가능한 Colab 학습 노트북 — 소형 드론 탐지 (Phase-1)

무료 티어의 12시간 세션 한도·유휴 타임아웃을 전제로, **중단되어도 마지막 epoch에서 자동 재개**되도록 만든 템플릿입니다.

## 사전 준비 (한 번만)

1. **데이터셋을 단일 아카이브로 묶어 Drive에 업로드**합니다.
   ```bash
   # 로컬(D드라이브)에서 실행 — 1280x736으로 materialize된 데이터셋 기준
   tar -cf external_uav_phase1_v2_1280x736.tar external_uav_phase1_v2_1280x736/
   ```
   > ⚠️ **압축하지 말고 `tar`만 쓰세요.** 이미 JPEG이라 gzip은 시간만 잡아먹고 거의 안 줄어듭니다.

2. Drive에 다음 구조를 만듭니다.
   ```
   MyDrive/drone_hackathon/
     ├─ data/external_uav_phase1_v2_1280x736.tar
     └─ runs/          (자동 생성됨 — 체크포인트가 여기 쌓입니다)
   ```

## 실행 방법

- **처음 실행**: 아래 셀을 위에서부터 순서대로 실행
- **세션이 끊긴 뒤**: 똑같이 위에서부터 다시 실행하면 됩니다. Drive의 `last.pt`를 찾아 **자동으로 이어서 학습**합니다
- **R9 → R10 전환**: `CFG`의 `RUN_NAME`과 `MODEL`만 바꾸고 재실행

## 이 노트북이 해결하는 것

| 문제 | 대응 |
|---|---|
| Drive I/O가 느려 GPU가 논다 | 단일 tar를 `/content`(로컬 SSD)로 복사 후 해제 |
| 세션이 죽으면 학습이 날아간다 | 매 epoch `last.pt`/`best.pt`를 Drive에 동기화 |
| `best.pt`가 mAP50-95 기준으로 뽑힌다 | **`best_map50.pt`를 별도로 추적·저장** (아래 §콜백 참조) |
| 재개 시 이전 설정을 잊는다 | `resume=True`가 체크포인트의 args를 복원 |

## 0. Colab 런타임 설정 (시작 전 1회)

**런타임 > 런타임 유형 변경**

| 항목 | 값 | 이유 |
|---|---|---|
| 하드웨어 가속기 | **L4 GPU** | 아래 참고 |
| 백그라운드 실행 | **켬** | Pro 핵심 혜택. 브라우저를 닫아도 학습 지속 |

### GPU를 L4로 하는 이유

YOLO11n은 파라미터 3.2M의 초경량 모델이라 1280 해상도에서도 **병목이 GPU 연산이 아니라
이미지 디코딩/증강(CPU)** 쪽입니다. A100을 붙여도 비례해서 빨라지지 않는데
CU는 2배 이상 소모됩니다. Pro의 100 CU로 R9와 R10 **두 번**을 돌려야 하므로 L4가 맞습니다.

- 학습 시작 후 첫 에폭 로그에서 GPU 사용률이 낮고 dataloader가 느리면 `WORKERS`를 올리세요.
- 우측 상단 리소스 표시기에서 **CU 잔량**을 주기적으로 확인하십시오.

### 실행 순서

1. `SMOKE = True`로 전체 셀 실행 (2 에폭)
2. **런타임 > 세션 관리 > 종료** 후 다시 전체 실행 → **3 에폭째부터 이어지는지 확인**
3. 확인되면 `SMOKE = False`, `BATCH`를 스모크에서 나온 정수로 고정, 전체 재실행

2번을 건너뛰지 마십시오. 재개가 실제로 되는지 확인되지 않은 체크포인트는 없는 것과 같습니다.

---


## 1. 설정 — 여기만 수정하면 됩니다

In [ ]:
# ── 스모크 테스트 스위치 ─────────────────────────────────────────────
# True  : 2 에폭만 돌려 "파이프라인 + 재개"를 검증합니다 (본런 전 필수)
#         RUN_NAME에 _smoke가 붙어 본런 체크포인트와 완전히 분리됩니다.
# False : 본런
SMOKE = True

_RUN  = "R9_yolo11n_1280x736"          # R10 : "R10_yolo11n_p2_1280x736"
_MODEL = "yolo11n.pt"                  # R10 : 팀 configs/model/p2_1280x736.yaml 경로

CFG = {
    # ---- 실험 식별 ----
    "RUN_NAME": _RUN + ("_smoke" if SMOKE else ""),
    "MODEL":    _MODEL,

    # ---- 입력 ----
    # 데이터셋을 1280x736으로 미리 materialize 해둔 상태를 전제합니다.
    # rect=True + imgsz=긴변(1280) 조합은 R1~R8과 동일한 방식입니다.
    "IMGSZ": 1280,
    "RECT":  True,

    # ---- 학습 (R1~R8과 동일하게 유지해야 공정 비교가 됩니다) ----
    "EPOCHS":   2 if SMOKE else 80,
    "PATIENCE": 20,

    # ⚠️ BATCH — 본런은 반드시 "고정 정수"로 돌리십시오.
    #    스모크(-1)에서 자동 결정된 값이 로그에 찍힙니다. 그 값을 여기 박고 본런하세요.
    #    자동(-1)인 채로 본런하면: A100에서 배치 48로 시작 -> 세션 끊김
    #    -> 재개 시 L4로 재배정 -> 배치 48이 그대로 복원 -> OOM으로 사망.
    #    (resume은 체크포인트에 저장된 batch를 그대로 씁니다. GPU를 다시 보지 않습니다.)
    "BATCH":    -1,

    "SEED":     42,
    "OPTIMIZER": "AdamW",
    "LR0": 0.001,
    "LRF": 0.01,

    # Colab Pro L4는 보통 vCPU 8. 1280 JPEG 디코딩이 무거워 2개로는 GPU가 굶습니다.
    # 스모크에서 RAM 경고가 뜨면 4로 낮추세요.
    "WORKERS": 8,

    # ---- 증강 (소형 객체 보존: mosaic 계열 off) ----
    "MOSAIC": 0.0, "MIXUP": 0.0, "COPY_PASTE": 0.0, "ERASING": 0.0,
    "SCALE": 0.2, "TRANSLATE": 0.05, "FLIPLR": 0.5,

    # ---- 경로 ----
    "DRIVE_ROOT":    "/content/drive/MyDrive/drone_hackathon",
    "DATA_ARCHIVE":  "data/external_uav_phase1_v2_1280x736.tar",
    "DATA_YAML_REL": "external_uav_phase1_v2_1280x736/data.yaml",
    "LOCAL_ROOT":    "/content/work",

    # ---- 동기화 ----
    "SYNC_EVERY": 1,     # N epoch마다 Drive에 체크포인트 동기화
}

if SMOKE:
    print("*" * 62)
    print("*  스모크 테스트 모드 — 2 에폭만 돕니다.")
    print("*  검증 후 SMOKE=False + BATCH를 고정값으로 바꾸고 본런하세요.")
    print("*" * 62)

for k, v in CFG.items():
    print(f"  {k:<14} {v}")


## 2. 환경 점검

GPU가 무엇인지, VRAM이 얼마인지 먼저 확인합니다. **T4가 아닌 L4/A100이 걸리면 훨씬 빠릅니다.**

In [ ]:
import subprocess, time, os

SESSION_START = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print(f"  CPU 코어: {os.cpu_count()}")
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout.strip())

# GPU가 안 붙었으면 여기서 멈추는 게 낫습니다
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU 선택"
print(f"\n  torch {torch.__version__} / CUDA {torch.version.cuda}")

## 3. Drive 마운트 + 패키지 설치

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = CFG["DRIVE_ROOT"]
CKPT_DRIVE = f"{DRIVE}/runs/{CFG['RUN_NAME']}"
os.makedirs(f"{CKPT_DRIVE}/weights", exist_ok=True)
print("체크포인트 위치:", CKPT_DRIVE)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 4. 데이터 준비 — Drive → 로컬 디스크

> 🔴 **Drive를 마운트한 채로 직접 학습하면 안 됩니다.** 수만 개 작은 파일을 Drive에서 읽으면 GPU가 대부분 놀게 됩니다. 단일 tar를 `/content`로 옮긴 뒤 해제합니다.

이미 해제되어 있으면 건너뜁니다 (같은 세션에서 셀을 다시 돌릴 때).

In [ ]:
import shutil, tarfile, glob, yaml
from pathlib import Path

LOCAL = Path(CFG["LOCAL_ROOT"]); LOCAL.mkdir(parents=True, exist_ok=True)
DATA_YAML = LOCAL / CFG["DATA_YAML_REL"]

if DATA_YAML.exists():
    print("이미 준비됨 — 건너뜁니다:", DATA_YAML)
else:
    src = Path(DRIVE) / CFG["DATA_ARCHIVE"]
    assert src.exists(), f"Drive에 아카이브가 없습니다: {src}"
    size_gb = src.stat().st_size / 2**30
    print(f"아카이브 {size_gb:.2f} GB")

    t0 = time.time()
    local_tar = LOCAL / src.name
    shutil.copy2(src, local_tar)                       # Drive -> 로컬 (단일 큰 파일이라 빠름)
    print(f"  복사 완료 {time.time()-t0:.0f}s")

    t0 = time.time()
    with tarfile.open(local_tar) as tf:
        tf.extractall(LOCAL)
    local_tar.unlink()                                  # 공간 회수
    print(f"  해제 완료 {time.time()-t0:.0f}s")

assert DATA_YAML.exists(), f"data.yaml을 찾을 수 없습니다: {DATA_YAML}"

# data.yaml의 path를 로컬 절대경로로 교정 (로컬에서 만든 yaml의 경로가 달라도 동작하도록)
d = yaml.safe_load(DATA_YAML.read_text())
d["path"] = str(DATA_YAML.parent)
DATA_YAML.write_text(yaml.safe_dump(d, allow_unicode=True, sort_keys=False))
print("\ndata.yaml:", d)

for split in ("train", "val", "test"):
    p = DATA_YAML.parent / d.get(split, "") if d.get(split) else None
    if p and p.exists():
        n = len(glob.glob(str(p / "**" / "*.*"), recursive=True))
        print(f"  {split:<6} {n:,} files")

## 5. 콜백 — Drive 동기화 + `best_map50.pt` 추적

**왜 `best_map50.pt`가 필요한가:** Ultralytics의 `best.pt`는 fitness = `0.1×mAP50 + 0.9×mAP50-95` 기준으로 선택됩니다. 그런데 우리 hard constraint는 **mAP@0.5**입니다. 실제로 R1에서 mAP50 최고였던 epoch 16 대신 epoch 14가 저장되어 16의 가중치를 잃었습니다.

아래 콜백이 **mAP50 최고 시점의 가중치를 따로 보관**합니다. `save_period=1`로 매 epoch을 전부 저장하는 것보다 훨씬 가볍습니다.

In [ ]:
STATE = {"best_map50": -1.0, "best_epoch": -1}

# 재개 시 이전 기록 복원
_meta = Path(CKPT_DRIVE) / "best_map50.txt"
if _meta.exists():
    try:
        v, e = _meta.read_text().split(",")
        STATE.update(best_map50=float(v), best_epoch=int(e))
        print(f"이전 최고 mAP50 복원: {STATE['best_map50']:.4f} (epoch {STATE['best_epoch']})")
    except Exception as ex:
        print("복원 실패, 새로 시작:", ex)


def _copy(src, dst):
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)


def sync_to_drive(trainer):
    """on_model_save 시점 — 이 시점의 last.pt가 현재 epoch 가중치입니다."""
    ep = int(getattr(trainer, "epoch", 0)) + 1
    sd = str(trainer.save_dir)

    # 1) mAP50 최고 갱신 여부 확인
    m = (getattr(trainer, "metrics", None) or {}).get("metrics/mAP50(B)")
    if m is not None and float(m) > STATE["best_map50"]:
        STATE.update(best_map50=float(m), best_epoch=ep)
        _copy(f"{sd}/weights/last.pt", f"{CKPT_DRIVE}/weights/best_map50.pt")
        (Path(CKPT_DRIVE) / "best_map50.txt").write_text(f"{STATE['best_map50']},{ep}")
        print(f"  [best_map50 갱신] epoch {ep}  mAP50={m:.4f}")

    # 2) 주기적 동기화
    if ep % CFG["SYNC_EVERY"] == 0:
        for rel in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml"):
            _copy(f"{sd}/{rel}", f"{CKPT_DRIVE}/{rel}")
        el = (time.time() - SESSION_START) / 3600
        print(f"  [sync] epoch {ep} -> Drive   (세션 경과 {el:.2f}h)")

print("콜백 준비 완료")

## 6. 학습 — 신규 시작 / 자동 재개

Drive에 `last.pt`가 있으면 그것을 로컬로 되돌린 뒤 `resume=True`로 이어서 학습합니다.

In [ ]:
from ultralytics import YOLO

PROJECT = str(LOCAL / "runs")
RUN_DIR = Path(PROJECT) / CFG["RUN_NAME"]
drive_last = Path(CKPT_DRIVE) / "weights" / "last.pt"
RESUME = drive_last.exists()

if RESUME:
    print("=" * 60)
    print("이전 체크포인트 발견 — 이어서 학습합니다")
    print("=" * 60)
    (RUN_DIR / "weights").mkdir(parents=True, exist_ok=True)
    for rel in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml"):
        _copy(f"{CKPT_DRIVE}/{rel}", str(RUN_DIR / rel))
    import torch as _t
    _ck = _t.load(str(RUN_DIR / "weights" / "last.pt"), map_location="cpu", weights_only=False)
    print(f"  마지막 완료 epoch: {_ck.get('epoch')}")
    del _ck
    model = YOLO(str(RUN_DIR / "weights" / "last.pt"))
    model.add_callback("on_model_save", sync_to_drive)
    results = model.train(resume=True)
else:
    print("=" * 60)
    print("신규 학습 시작")
    print("=" * 60)
    model = YOLO(CFG["MODEL"])
    model.add_callback("on_model_save", sync_to_drive)
    results = model.train(
        data=str(DATA_YAML),
        project=PROJECT, name=CFG["RUN_NAME"], exist_ok=True,
        imgsz=CFG["IMGSZ"], rect=CFG["RECT"],
        epochs=CFG["EPOCHS"], patience=CFG["PATIENCE"], batch=CFG["BATCH"],
        seed=CFG["SEED"], deterministic=True,
        optimizer=CFG["OPTIMIZER"], lr0=CFG["LR0"], lrf=CFG["LRF"], cos_lr=True,
        workers=CFG["WORKERS"], amp=True,
        mosaic=CFG["MOSAIC"], mixup=CFG["MIXUP"],
        copy_paste=CFG["COPY_PASTE"], erasing=CFG["ERASING"],
        scale=CFG["SCALE"], translate=CFG["TRANSLATE"], fliplr=CFG["FLIPLR"],
        plots=True, val=True,
    )

## 7. 최종 동기화 및 결과 요약

In [ ]:
import pandas as pd

for rel in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml", "results.png"):
    _copy(str(RUN_DIR / rel), f"{CKPT_DRIVE}/{rel}")
print("최종 동기화 완료 ->", CKPT_DRIVE)

csv = RUN_DIR / "results.csv"
if csv.exists():
    df = pd.read_csv(csv); df.columns = [c.strip() for c in df.columns]
    col = next((c for c in df.columns if "mAP50" in c and "95" not in c), None)
    col95 = next((c for c in df.columns if "mAP50-95" in c), None)
    print(f"\n총 epoch: {len(df)}")
    if col:
        i = df[col].idxmax()
        print(f"  최고 mAP50    : {df[col].max():.4f}  (epoch {i+1})")
    if col95:
        j = df[col95].idxmax()
        print(f"  최고 mAP50-95 : {df[col95].max():.4f}  (epoch {j+1})  <- best.pt는 이쪽")
    print(f"  추적된 best_map50: {STATE['best_map50']:.4f} (epoch {STATE['best_epoch']})")
    print("\n마지막 5 epoch:")
    print(df.tail(5)[[c for c in df.columns if "mAP" in c or c == "epoch"]].to_string(index=False))
    # 학습 곡선이 끝에서 아직 상승 중이면 epoch를 늘릴 여지가 있습니다 (특히 P2)
    if col and len(df) >= 10:
        tail = df[col].tail(10)
        trend = "상승 중 (epoch 늘릴 여지 있음)" if tail.iloc[-1] > tail.iloc[0] else "정체/하락"
        print(f"\n  최근 10 epoch 추세: {trend}")

## 8. (선택) 고정 test 평가 — 스케일 구간별 Recall 포함

전체 mAP만 보면 **DD(큰 드론)에 희석되어** 1280의 효과가 안 보입니다. 기업 데이터는 median 9×4 px이므로 **작은 구간의 Recall**이 실제 판정 기준입니다.

In [ ]:
import numpy as np

WEIGHTS = f"{CKPT_DRIVE}/weights/best_map50.pt"   # mAP50 기준 최고 가중치
if not os.path.exists(WEIGHTS):
    WEIGHTS = str(RUN_DIR / "weights" / "best.pt")
print("평가 가중치:", WEIGHTS)

m = YOLO(WEIGHTS)
metrics = m.val(data=str(DATA_YAML), split="test", imgsz=CFG["IMGSZ"],
                rect=CFG["RECT"], plots=False)
print(f"\ntest  P {metrics.box.mp:.4f} / R {metrics.box.mr:.4f} / "
      f"mAP50 {metrics.box.map50:.4f} / mAP50-95 {metrics.box.map:.4f}")

# ---- GT 라벨 기준 스케일 구간 분포 (판정 기준선 확인용) ----
lbl_dir = Path(d["path"]) / d["test"].replace("images", "labels")
if lbl_dir.exists():
    W, H = CFG["IMGSZ"], int(CFG["IMGSZ"] * 736 / 1280)
    sizes = []
    for f in glob.glob(str(lbl_dir / "*.txt")):
        for line in open(f):
            p = line.split()
            if len(p) >= 5:
                sizes.append(max(float(p[3]) * W, float(p[4]) * H))
    if sizes:
        sizes = np.array(sizes)
        print(f"\ntest GT 박스 {len(sizes):,}개 — max(w,h) 중앙값 {np.median(sizes):.1f} px")
        for nm, lo, hi in [("ultra_tiny <8", 0, 8), ("tiny 8-16", 8, 16),
                           ("small 16-32", 16, 32), ("medium 32-96", 32, 96),
                           ("large >=96", 96, 1e9)]:
            c = int(((sizes >= lo) & (sizes < hi)).sum())
            print(f"  {nm:<16} {c:>7,}  ({c/len(sizes)*100:5.1f}%)")
        print("\n  → 구간별 Recall은 별도 스크립트(predictions.json 재집계)로 계산하십시오.")
        print("    R9 vs R10 판정은 ultra_tiny/tiny 구간에서 갈립니다.")

---

## 세션이 끊겼을 때

**그냥 노트북을 다시 열고 1번 셀부터 순서대로 실행하면 됩니다.** 6번 셀이 Drive의 `last.pt`를 찾아 자동으로 이어서 학습합니다.

매 재개마다 데이터 복사·해제(5~15분)가 반복되는데, 이게 무료 티어의 숨은 비용입니다.

## 산출물

```
MyDrive/drone_hackathon/runs/<RUN_NAME>/
  ├─ weights/last.pt          재개용
  ├─ weights/best.pt          Ultralytics 기본 (mAP50-95 기준) 
  ├─ weights/best_map50.pt    ★ mAP50 기준 — 우리 hard constraint에 맞는 가중치
  ├─ best_map50.txt           최고값과 epoch 기록
  ├─ results.csv / results.png
  └─ args.yaml
```

## 체크리스트

- [ ] R9(`yolo11n.pt`)와 R10(`yolo11n-p2.yaml`) **둘 다** 실행 — 1280에서 P2가 역전하는지가 현재 핵심 질문입니다
- [ ] `seed=42`, `epochs=80`, `patience=20`을 R1~R8과 동일하게 유지 (공정 비교)
- [ ] 실제 batch 크기를 로그에서 기록 (`batch=-1` 자동 결정값)
- [ ] P2는 **가중치 transfer 로그**를 남길 것 — P2 브랜치는 새로 초기화되므로 미수렴 여부 판별에 필요합니다
- [ ] 학습 곡선이 끝에서 상승 중이면 epoch 확대 검토

> ⚠️ **`yolo11n-p2.yaml`은 Ultralytics 기본 제공이 아닐 수 있습니다.** R7·R8에서 쓰신 팀의 P2 config를 Drive에 올려두고 `CFG["MODEL"]`에 그 경로를 지정하십시오. 사전학습 가중치는 P2 브랜치에 1:1 대응이 없어 새로 초기화되지만, fine-tuning 자체는 정상 동작합니다.